<a href="https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M02/M02_Lab1_Prompting_Techniques.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 30px 36px; border-radius: 14px; margin-bottom: 20px;">
  <h1 style="color: #fff; margin: 0; font-size: 28px;">🎯 M02 Lab 1 — Prompting Techniques</h1>
  <p style="color: rgba(255,255,255,0.85); margin: 8px 0 0; font-size: 15px;">DADS 5250: Generative AI in Practice &nbsp;|&nbsp; Prof. Dehghani</p>
  <p style="color: rgba(255,255,255,0.65); margin: 6px 0 0; font-size: 13px;">⭐ Difficulty: Beginner &nbsp;|&nbsp; ⏱ Time: ~10 min</p>
</div>

<div style="background: #f0f4ff; border-left: 4px solid #0055d4; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="color: #001a70; margin: 0 0 8px;">🎯 Learning Objectives</h3>
  <ol style="margin: 0; color: #1a1a2e; font-size: 14px;">
    <li>Apply <b>zero-shot</b>, <b>few-shot</b>, and <b>chain-of-thought (CoT)</b> prompting</li>
    <li>Understand when each technique works best and why</li>
    <li>Compare output quality across techniques side by side</li>
    <li>Practice writing effective few-shot prompts for real tasks</li>
  </ol>
</div>

In [ ]:
# ============================================================
# 📦 Install Dependencies
# ============================================================
!pip install -q --upgrade openai tiktoken
!pip install -q git+https://github.com/mdehghani86/DADS5250-GenAI.git#subdirectory=utils

In [ ]:
# ============================================================
# 🔑 API Key & Client Setup
# ============================================================
from openai import OpenAI
from dads5250 import setup_openai, pp, show_response, show_expected, show_info, compare_responses, count_tokens, estimate_cost, quiz

# This reads your OPENAI_API_KEY from Colab Secrets automatically
client = setup_openai()

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">1️⃣ Zero-Shot Prompting</h2>
</div>

Ask the model to perform a task **without any examples**. It relies entirely on its pre-training knowledge.

This is the simplest approach — you just describe what you want. It works surprisingly well for straightforward tasks like classification, summarization, and translation.

In [ ]:
# ============================================================
# 🎯 Zero-Shot: Sentiment Classification
# ============================================================

reviews = [
    "The food was decent but the service was painfully slow.",
    "Absolutely stunning views and the staff went above and beyond!",
    "It was okay. Nothing special, nothing terrible.",
]

for review in reviews:
    r = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "user", "content": f"Classify the sentiment of this review as positive, negative, or neutral. Reply with just the label.\n\n'{review}'"}
        ],
        max_tokens=10,
    )
    label = r.choices[0].message.content.strip()
    print(f"📝 \"{review}\"")
    print(f"   → Sentiment: {label}\n")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">2️⃣ Few-Shot Prompting</h2>
</div>

Provide **examples** of the desired input → output pattern to guide the model's format and behavior.

Few-shot is powerful because the model learns the pattern *from your examples* — you're essentially "programming" it with demonstrations rather than instructions.

In [ ]:
# ============================================================
# 📋 Few-Shot: Sentiment Classification (Same Task, Better Control)
# ============================================================

few_shot_prompt = """Classify the sentiment as positive, negative, or neutral.

Review: "Absolutely loved the atmosphere and the pasta was divine!"
Sentiment: positive

Review: "The room was dirty and the staff was rude."
Sentiment: negative

Review: "It was an average experience, nothing special."
Sentiment: neutral

Review: "The food was decent but the service was painfully slow."
Sentiment:"""

r = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "user", "content": few_shot_prompt}],
    max_tokens=10,
)

print(f"🎯 Few-shot result: {r.choices[0].message.content.strip()}")
print(f"\n📊 Notice: the model follows the exact format from our examples!")

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">3️⃣ Chain-of-Thought (CoT) Prompting</h2>
</div>

Ask the model to **think step by step** before answering. This dramatically improves reasoning accuracy on math, logic, and multi-step problems.

The magic phrase: *"Think step by step."* — Let's see the difference it makes:

In [ ]:
# ============================================================
# 🧠 CoT: Math Problem — With vs Without
# ============================================================

math_problem = "A store has 45 apples. They sell 12 in the morning and receive a shipment of 30. Then they sell 18 in the afternoon. How many apples are left?"

# ❌ Without CoT
r_no_cot = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "user", "content": math_problem}],
    max_tokens=100,
)

# ✅ With CoT
r_cot = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "user", "content": math_problem + "\n\nThink step by step."}],
    max_tokens=300,
)

compare_responses({
    "❌ Without CoT": r_no_cot.choices[0].message.content,
    "✅ With CoT": r_cot.choices[0].message.content,
})

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">📝 Knowledge Check</h2>
</div>

In [ ]:
# ============================================================
# 📝 Quiz — Test Your Understanding
# ============================================================

quiz([
    {
        "q": "What is the key difference between zero-shot and few-shot prompting?",
        "options": [
            "Zero-shot uses GPT, few-shot uses Gemini",
            "Zero-shot provides no examples, few-shot provides examples of the desired pattern",
            "Zero-shot is faster, few-shot is slower",
            "There is no difference"
        ],
        "answer": 1,
        "explanation": "Few-shot includes examples in the prompt to guide the model's output format and behavior. Zero-shot relies entirely on instructions."
    },
    {
        "q": "When is chain-of-thought (CoT) prompting most useful?",
        "options": [
            "For simple factual questions",
            "For generating creative fiction",
            "For multi-step reasoning and math problems",
            "For making API calls faster"
        ],
        "answer": 2,
        "explanation": "CoT helps the model break down complex problems into steps, greatly improving accuracy on reasoning tasks."
    },
    {
        "q": "What phrase typically triggers chain-of-thought reasoning?",
        "options": [
            "\"Be creative\"",
            "\"Think step by step\"",
            "\"Use JSON format\"",
            "\"Be concise\""
        ],
        "answer": 1,
        "explanation": "\"Think step by step\" is the classic CoT trigger. It tells the model to show its reasoning process before giving a final answer."
    }
])

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 18px 24px; border-radius: 10px; margin: 20px 0 12px;">
  <h2 style="color: white; margin: 0; font-size: 20px;">🔧 Hands-On Exercise</h2>
</div>

### Exercise 1: Write a Few-Shot Extraction Prompt

Create a few-shot prompt that extracts the **product name** and **price** from product descriptions. Complete the pattern below:

# ============================================================
# 🔧 Exercise 1: Few-Shot Product Extraction
# ============================================================
# Complete the few-shot prompt — fill in the blanks (-----)

extraction_prompt = """Extract the product name and price from each description.

Description: "The Sony WH-1000XM5 headphones are available for $348."
Product: Sony WH-1000XM5 | Price: $348

Description: "Get the new MacBook Air M3 starting at $1,099."
Product: MacBook Air M3 | Price: $1,099

Description: "The Kindle Paperwhite is now on sale for $129.99."
Product: -----  | Price: -----"""
# 👆 YOUR CODE: The model should complete this — but what pattern will it follow?

response_ex1 = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "-----", "content": extraction_prompt}],  # What role sends the prompt?
    max_tokens=50,
)

show_response(response_ex1)

In [ ]:
# --- Test ---
output = response_ex1.choices[0].message.content
assert "Kindle" in output or "kindle" in output, "Expected the model to extract 'Kindle Paperwhite'"
show_expected("Product: Kindle Paperwhite | Price: $129.99")

<div style="background: #e8f5e9; border-left: 4px solid #4caf50; padding: 14px 18px; border-radius: 0 8px 8px 0; margin: 12px 0;">
  <h3 style="margin: 0 0 8px;">📖 Self-Study Activity</h3>
  <p style="margin: 0; font-size: 14px;">Try these experiments on your own:</p>
  <ol style="font-size: 14px;">
    <li>Take the email categorization example and add a 5th category — does the model handle it?</li>
    <li>Try few-shot with <b>wrong examples</b> (e.g., label positive reviews as "negative") — does the model follow your examples or its own judgment?</li>
    <li>Combine few-shot + CoT: show an example with step-by-step reasoning, then give a new problem</li>
  </ol>
</div>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 24px 30px; border-radius: 12px; margin: 24px 0 0; text-align: center;">
  <h2 style="color: white; margin: 0 0 10px;">🎉 Lab 1 Complete!</h2>
  <p style="color: rgba(255,255,255,0.85); margin: 0; font-size: 14px;">You've mastered the three core prompting techniques that form the foundation of prompt engineering.</p>
  <div style="background: rgba(255,255,255,0.15); border-radius: 8px; padding: 12px; margin-top: 14px; text-align: left;">
    <p style="color: white; margin: 0 0 6px; font-weight: bold; font-size: 14px;">Key Takeaways:</p>
    <ul style="color: rgba(255,255,255,0.9); margin: 0; font-size: 13px;">
      <li><b>Zero-shot</b> — no examples needed, works for straightforward tasks</li>
      <li><b>Few-shot</b> — include examples to control format and improve accuracy</li>
      <li><b>Chain-of-thought</b> — "Think step by step" for reasoning and math</li>
      <li>Techniques can be <b>combined</b> for even better results</li>
    </ul>
  </div>
  <p style="color: rgba(255,255,255,0.7); margin: 14px 0 0; font-size: 13px;"><b>Next →</b> M02 Lab 2: System Messages, Templates &amp; Model Comparison</p>
</div>

In [ ]:
# ============================================================
# 🔧 Exercise 2: Fix with CoT
# ============================================================

tricky_problem = """A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. 
How much does the ball cost?"""

# 👆 YOUR CODE: Add the CoT trigger to the prompt below
response_ex2 = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[{"role": "user", "content": tricky_problem + "\n\n-----"}],  # Add CoT trigger here
    max_tokens=200,
)

show_response(response_ex2)

In [ ]:
# --- Test ---
answer = response_ex2.choices[0].message.content.lower()
assert "0.05" in answer or "5 cents" in answer or "$0.05" in answer, "The answer should be $0.05 (not $0.10!)"
show_expected("The ball costs $0.05 (not $0.10 — that's the common intuitive but wrong answer!)")

**📝 Your Observations:** *(double-click to edit this cell)*

1. Did the model get the bat-and-ball problem right with CoT? _[Your answer]_

2. What happened when you ran it without "Think step by step"? _[Your answer]_

3. Why does CoT help with this specific problem? _[Your answer]_

---
## 4. Knowledge Check

In [ ]:
quiz([
    {
        "q": "What is the key difference between zero-shot and few-shot prompting?",
        "options": [
            "Zero-shot uses GPT, few-shot uses Gemini",
            "Zero-shot provides no examples, few-shot provides examples of the desired pattern",
            "Zero-shot is faster, few-shot is slower",
            "There is no difference"
        ],
        "answer": 1,
        "explanation": "Few-shot includes examples in the prompt to guide the model's output format and behavior."
    },
    {
        "q": "When is chain-of-thought (CoT) prompting most useful?",
        "options": [
            "For simple factual questions",
            "For generating creative fiction",
            "For multi-step reasoning and math problems",
            "For making API calls faster"
        ],
        "answer": 2,
        "explanation": "CoT helps the model break down complex problems into steps, greatly improving accuracy on reasoning tasks."
    }
])

---
## 5. Hands-on: Write a Few-Shot Prompt

Create a few-shot prompt that extracts the **product name** and **price** from product descriptions.

In [ ]:
# Complete the few-shot extraction prompt

response_ex1 = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "user", "content": """Extract the product name and price from the description.

Description: "The Sony WH-1000XM5 headphones are available for $348."
Product: Sony WH-1000XM5 | Price: $348

Description: "Get the new MacBook Air M3 starting at $1,099."
Product: MacBook Air M3 | Price: $1,099

Description: "The Kindle Paperwhite is now on sale for $129.99."
Product: _____  | Price: _____"""}
    ]  # YOUR CODE HERE: complete the pattern above, then try adding a new test description
)

show_response(response_ex1)

In [ ]:
# --- Test ---
output = response_ex1.choices[0].message.content
assert "Kindle" in output or "kindle" in output, "Expected the model to extract 'Kindle Paperwhite'"
show_expected("Product: Kindle Paperwhite | Price: $129.99")

---
## Summary

- **Zero-shot**: No examples — works for straightforward tasks
- **Few-shot**: Include examples to guide format and accuracy
- **Chain-of-thought**: "Think step by step" for reasoning tasks

**Next:** M02 Lab 2 — System Messages, Templates & Model Comparison